In [1]:
import numpy as np
import pandas as pd

# Task 0
Read the dataset from csv file & perform data cleaning - remove all rows, which contains `?` in some columns.
Also check for data correctness (salary & salary $K).

In [2]:
from pathlib import Path

csv_path = Path("data/adult.csv")
if not csv_path.exists():
    csv_path = Path("../data/adult.csv")

df = pd.read_csv(csv_path, index_col=0)
df = df.replace("?", pd.NA).dropna()

salary_ok = (
    (df["salary"].eq("<=50K") & df["salary K$"].le(50))
    | (df["salary"].eq(">50K") & df["salary K$"].gt(50))
)
print("All salary / salary K$ rows are consistent:", bool(salary_ok.all()))
df = df.loc[salary_ok]
df

All salary / salary K$ rows are consistent: True


,age,workclass,education,marital-status,occupation,relationship,race,sex,hours-per-week,native-country,salary,salary K$
0,39,State-gov,Bachelors,Never-married,Adm-clerical,Not-in-family,White,Male,40,United-States,<=50K,39
1,50,Self-emp-not-inc,Bachelors,Married-civ-spouse,Exec-managerial,Husband,White,Male,13,United-States,<=50K,35
2,38,Private,HS-grad,Divorced,Handlers-cleaners,Not-in-family,White,Male,40,United-States,<=50K,27
3,53,Private,11th,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,40,United-States,<=50K,43
4,28,Private,Bachelors,Married-civ-spouse,Prof-specialty,Wife,Black,Female,40,Cuba,<=50K,25
...,...,...,...,...,...,...,...,...,...,...,...,...
32556,27,Private,Assoc-acdm,Married-civ-spouse,Tech-support,Wife,White,Female,38,United-States,<=50K,36
32557,40,Private,HS-grad,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,40,United-States,>50K,173
32558,58,Private,HS-grad,Widowed,Adm-clerical,Unmarried,White,Female,40,United-States,<=50K,40
32559,22,Private,HS-grad,Never-married,Adm-clerical,Own-child,White,Male,20,United-States,<=50K,38


# Task 1
Print the count of men and women in the dataset.

In [3]:
df["sex"].value_counts()

sex
Male      20380
Female     9782
Name: count, dtype: int64

# Task 2
Find the average age of men in dataset

In [4]:
df.loc[df["sex"].eq("Male"), "age"].mean()

np.float64(39.18400392541707)

# Task 3
Get the percentage of people from Poland (native-country)

In [5]:
df["native-country"].eq("Poland").mean() * 100

np.float64(0.18566408063125786)

# Task 4
Get the mean and standard deviation of the age for people who earn > 50K per year. After this, get it for those who earn <= 50K.

In [6]:
high_mask = df["salary"].eq(">50K")
low_mask = df["salary"].eq("<=50K")

print("Age for salary >50K:")
print(df.loc[high_mask, "age"].agg(["mean", "std"]))
print("\nAge for salary <=50K:")
print(df.loc[low_mask, "age"].agg(["mean", "std"]))

Age for salary >50K:
mean    43.959110
std     10.269633
Name: age, dtype: float64

Age for salary <=50K:
mean    36.608060
std     13.464631
Name: age, dtype: float64


# Task 5
Check, if there are some people without higher education (education: Bachelors, Prof-school, Assoc-acdm, Assoc-voc, Masters, Doctorate), but with > 50K salary

In [7]:
higher_education = np.array(
    "Bachelors Prof-school Assoc-acdm Assoc-voc Masters Doctorate".split()
)

no_higher_rich = df.loc[
    ~df["education"].isin(higher_education) & df["salary"].eq(">50K")
]
print("People without higher education and salary >50K:", len(no_higher_rich))
print("Such people exist:", len(no_higher_rich) > 0)
no_higher_rich.head()

People without higher education and salary >50K: 3178
Such people exist: True


,age,workclass,education,marital-status,occupation,relationship,race,sex,hours-per-week,native-country,salary,salary K$
7,52,Self-emp-not-inc,HS-grad,Married-civ-spouse,Exec-managerial,Husband,White,Male,45,United-States,>50K,307
10,37,Private,Some-college,Married-civ-spouse,Exec-managerial,Husband,Black,Male,80,United-States,>50K,116
55,43,Private,Some-college,Married-civ-spouse,Tech-support,Husband,White,Male,40,United-States,>50K,341
67,53,Private,HS-grad,Married-civ-spouse,Adm-clerical,Wife,White,Female,40,United-States,>50K,225
68,49,Self-emp-inc,Some-college,Married-civ-spouse,Exec-managerial,Husband,White,Male,50,United-States,>50K,194


# Task 6
Get the statistics of age for each type of education. Use `groupby` and `describe` for this.

In [8]:
df.groupby("education")["age"].describe()

,count,mean,std,min,25%,50%,75%,max
education,,,,,,,,
10th,820.0,37.897561,16.225795,17.0,23.0,36.0,52.0,90.0
11th,1048.0,32.363550,15.089307,17.0,18.0,28.5,43.0,90.0
12th,377.0,32.013263,14.373710,17.0,19.0,28.0,41.0,79.0
1st-4th,151.0,44.622517,14.929051,19.0,33.0,44.0,56.0,81.0
5th-6th,288.0,41.649306,14.754622,17.0,28.0,41.0,53.0,82.0
7th-8th,557.0,47.631957,15.737479,17.0,34.0,49.0,60.0,90.0
9th,455.0,40.303297,15.335754,17.0,28.0,38.0,53.0,90.0
Assoc-acdm,1008.0,37.286706,10.509755,19.0,29.0,36.0,44.0,90.0
Assoc-voc,1307.0,38.246366,11.181253,19.0,30.0,37.0,45.0,84.0


# Task 7
Compare the married and non-married men salaries. Who earns more? (>50K or <=50K)
Married men are those, whom `marital-status` starts with "Married". Others are not.

In [9]:
men = df.loc[df["sex"].eq("Male")]
married_mask = men["marital-status"].str.startswith("Married")

married_mean_k = men.loc[married_mask, "salary K$"].mean()
non_married_mean_k = men.loc[~married_mask, "salary K$"].mean()
married_share_over_50k = men.loc[married_mask, "salary"].eq(">50K").mean()
non_married_share_over_50k = men.loc[~married_mask, "salary"].eq(">50K").mean()

print("Married men: mean salary K$ =", married_mean_k, "; share >50K =", married_share_over_50k)
print("Non-married men: mean salary K$ =", non_married_mean_k, "; share >50K =", non_married_share_over_50k)

married_predominant = np.where(married_share_over_50k > 0.5, ">50K", "<=50K").item()
non_married_predominant = np.where(non_married_share_over_50k > 0.5, ">50K", "<=50K").item()
print("Married men: predominant salary category is", married_predominant)
print("Non-married men: predominant salary category is", non_married_predominant)
print(
    "Married men earn more (higher share of >50K)"
    if married_share_over_50k > non_married_share_over_50k
    else "Non-married men earn more (higher share of >50K)"
)

Married men: mean salary K$ = 107.49455968688845 ; share >50K = 0.4479843444227006
Non-married men: mean salary K$ = 46.59723865877712 ; share >50K = 0.08849441157133466
Married men: predominant salary category is <=50K
Non-married men: predominant salary category is <=50K
Married men earn more (higher share of >50K)


# Task 8
Get the max hours per week some person works. How many people works the same amount of hours per week?

In [10]:
max_hours = df["hours-per-week"].max()
people_at_max = df["hours-per-week"].eq(max_hours).sum()
print("Max hours per week:", max_hours)
print("People working that many hours:", people_at_max)

Max hours per week: 99
People working that many hours: 78


# Task 9
Analyze the correlation between data in dataset. Understand connected fields in it and print highlight thier connection.

In [11]:
corr_df = df[["age", "hours-per-week", "salary K$"]].copy()
corr_df["salary_over_50k"] = df["salary"].eq(">50K").astype(int)
corr_matrix = corr_df.corr()
print(corr_matrix)
print(
    "\nHighlights:\n"
    "- salary K$ and salary_over_50k are strongly positively correlated, "
    "as expected from how the binary salary label is defined.\n"
    "- age has a weak positive correlation with salary K$ and salary_over_50k: "
    "older people tend to earn slightly more on average.\n"
    "- hours-per-week has a weak positive correlation with salary measures: "
    "people who work more hours tend to earn a bit more."
)

                      age  hours-per-week  salary K$  salary_over_50k
age              1.000000        0.101599   0.208203         0.241998
hours-per-week   0.101599        1.000000   0.196378         0.229480
salary K$        0.208203        0.196378   1.000000         0.853894
salary_over_50k  0.241998        0.229480   0.853894         1.000000

Highlights:
- salary K$ and salary_over_50k are strongly positively correlated, as expected from how the binary salary label is defined.
- age has a weak positive correlation with salary K$ and salary_over_50k: older people tend to earn slightly more on average.
- hours-per-week has a weak positive correlation with salary measures: people who work more hours tend to earn a bit more.
